# BeautifulSoup vs XPath: Cara Memilih Elemen HTML

**Dari yang paling mudah sampai yang kompleks — tiap case ditunjukkan dengan DUA cara:
BeautifulSoup dan XPath.**

Tujuannya: begitu paham *konsep memilih elemen*, kamu bisa pakai tool mana pun.

**Untuk siapa:** peserta yang sudah kenal HTML dasar & ingin mahir memilih elemen.

**Yang akan dibahas:** memilih berdasarkan **tag**, **class** (termasuk multi-class),
**id**, **atribut** (`data-*`, `aria-label`, `role`, `href`), pencocokan **sebagian**
(`contains`, `starts-with`, regex), pencocokan **teks**, navigasi **parent/sibling/child**,
sampai **kombinasi kondisi** dan filter **numerik**.

> **Catatan penting:** BeautifulSoup **tidak** mendukung XPath. Untuk XPath di HTML statis
> kita pakai **`lxml`** (`tree.xpath(...)`). lxml sudah jadi dependency project ini.
> (Di Selenium, XPath dipakai lewat `By.XPATH` — lihat notebook `walkthrough.ipynb`.)


## Outline (mudah → kompleks)

0. Setup — HTML contoh + objek `soup` (BS4) & `tree` (lxml)
1. Pilih berdasarkan **tag**
2. Pilih berdasarkan **id**
3. Pilih berdasarkan **class** (1 token)
4. **Multi-class** — perbedaan penting BS4 vs XPath ⚠️
5. Elemen **tanpa class** → pakai **posisi** (nth)
6. Pilih berdasarkan **atribut** (`data-*`)
7. **aria-label** & **role** (atribut aksesibilitas)
8. **href** — ambil semua link, link internal, mailto
9. Pencocokan **sebagian** — `contains`, `starts-with`, regex
10. Pilih berdasarkan **teks**
11. Navigasi: **parent / ancestor**
12. Navigasi: **sibling**
13. Navigasi: **direct child** (`/` vs `//`)
14. **Kombinasi** kondisi + filter **numerik**
15. Mini-project: ekstrak semua produk → list of dict (BS4 vs XPath)
16. Cheat sheet, pitfalls, latihan


In [1]:
import re

from bs4 import BeautifulSoup
from lxml import html as lxml_html

# HTML contoh: mirip halaman katalog e-commerce, sengaja dibuat beragam:
# - ada class tunggal & multi-class      - ada id
# - ada atribut data-*, aria-label, role - ada href internal/eksternal/mailto
# - ada elemen TANPA class               - ada produk dengan data tidak lengkap
HTML = """
<html>
  <body>
    <header id="top-nav" role="navigation" aria-label="Menu utama">
      <a href="/" class="logo">TokoKita</a>
      <ul>
        <li><a href="/kategori/buku">Buku</a></li>
        <li><a href="/kategori/elektronik">Elektronik</a></li>
        <li><a href="https://promo.external.com/diskon">Promo Eksternal</a></li>
      </ul>
    </header>

    <main id="content">
      <h1>Daftar Produk</h1>
      <p>Menampilkan beberapa produk pilihan.</p>

      <div class="product-list">
        <div class="product card featured" data-id="101" data-category="buku">
          <h2 class="product-name">Belajar Python</h2>
          <span class="price" data-price="75000">Rp75.000</span>
          <span class="rating" aria-label="Rating 4.5 dari 5">4.5</span>
          <a href="/produk/101" class="btn btn-primary" aria-label="Lihat Belajar Python">Lihat</a>
          <button aria-label="Tambah Belajar Python ke keranjang">Tambah</button>
        </div>

        <div class="product card" data-id="102" data-category="elektronik">
          <h2 class="product-name">Mouse Wireless</h2>
          <span class="price" data-price="150000">Rp150.000</span>
          <span class="rating" aria-label="Rating 4.0 dari 5">4.0</span>
          <a href="/produk/102" class="btn btn-primary" aria-label="Lihat Mouse Wireless">Lihat</a>
          <button aria-label="Tambah Mouse Wireless ke keranjang">Tambah</button>
        </div>

        <div class="product card" data-id="103" data-category="buku">
          <h2 class="product-name">Data Engineering 101</h2>
          <span class="price" data-price="120000">Rp120.000</span>
          <span class="rating" aria-label="Rating 5.0 dari 5">5.0</span>
          <a href="/produk/103" class="btn btn-primary" aria-label="Lihat Data Engineering 101">Lihat</a>
          <button aria-label="Tambah Data Engineering 101 ke keranjang">Tambah</button>
        </div>

        <div class="product card" data-id="104" data-category="elektronik">
          <h2 class="product-name">Keyboard Mekanik</h2>
          <span class="price" data-price="350000">Rp350.000</span>
          <span class="badge">Habis</span>
          <a href="/produk/104" class="btn btn-disabled" aria-label="Lihat Keyboard Mekanik">Lihat</a>
        </div>
      </div>

      <footer>
        <p>Kontak: <a href="mailto:cs@tokokita.id">cs@tokokita.id</a></p>
        <span>(c) 2026 TokoKita</span>
      </footer>
    </main>
  </body>
</html>
"""

# Dua objek untuk dua pendekatan:
soup = BeautifulSoup(HTML, "html.parser")   # untuk BeautifulSoup
tree = lxml_html.fromstring(HTML)           # untuk XPath (lxml)


def show(judul, items):
    """Bantu cetak hasil biar rapi & gampang dibandingkan."""
    print(f"{judul}  -> {len(items)} hasil")
    for it in items:
        print("   ", it)


print("soup & tree siap. Contoh dipakai sepanjang notebook ✅")


soup & tree siap. Contoh dipakai sepanjang notebook ✅


## 1. Pilih berdasarkan tag (paling mudah)

Ambil semua nama produk yang ada di tag `<h2>`.

| | Sintaks |
| --- | --- |
| **BS4** | `soup.find_all("h2")` |
| **XPath** | `//h2` (elemen) atau `//h2/text()` (langsung teksnya) |


In [2]:
# BS4: ambil elemen lalu teksnya
bs4_hasil = [h2.get_text(strip=True) for h2 in soup.find_all("h2")]
show("BS4  //h2", bs4_hasil)

# XPath: bisa langsung ambil teks dengan /text()
xpath_hasil = [t.strip() for t in tree.xpath("//h2/text()")]
show("XPath //h2/text()", xpath_hasil)

assert bs4_hasil == xpath_hasil  # hasilnya sama


BS4  //h2  -> 4 hasil
    Belajar Python
    Mouse Wireless
    Data Engineering 101
    Keyboard Mekanik
XPath //h2/text()  -> 4 hasil
    Belajar Python
    Mouse Wireless
    Data Engineering 101
    Keyboard Mekanik


## 2. Pilih berdasarkan id

`id` itu **unik** dalam satu halaman, jadi cara paling pasti untuk menunjuk satu elemen.

| | Sintaks |
| --- | --- |
| **BS4** | `soup.find(id="content")` |
| **XPath** | `//*[@id="content"]` |


In [3]:
# BS4
header_bs4 = soup.find(id="top-nav")
print("BS4  :", header_bs4.name, "| role =", header_bs4.get("role"))

# XPath ([0] karena xpath selalu mengembalikan list)
header_xp = tree.xpath('//*[@id="top-nav"]')[0]
print("XPath:", header_xp.tag, "| role =", header_xp.get("role"))


BS4  : header | role = navigation
XPath: header | role = navigation


## 3. Pilih berdasarkan class (1 token)

Ambil semua harga yang ada di `<span class="price">`.

| | Sintaks |
| --- | --- |
| **BS4** | `soup.find_all(class_="price")` (perhatikan ada garis bawah: `class_`) |
| **XPath** | `//span[@class="price"]` |

Di sini class-nya cuma satu kata (`price`), jadi dua-duanya gampang.
Hati-hati kalau class-nya lebih dari satu kata → lihat case berikutnya.

In [4]:
# BS4: class_ dengan garis bawah (karena 'class' kata kunci Python)
harga_bs4 = [s.get_text(strip=True) for s in soup.find_all(class_="price")]
show("BS4  class_='price'", harga_bs4)

# XPath
harga_xp = [s.text.strip() for s in tree.xpath('//span[@class="price"]')]
show("XPath //span[@class='price']", harga_xp)

BS4  class_='price'  -> 4 hasil
    Rp75.000
    Rp150.000
    Rp120.000
    Rp350.000
XPath //span[@class='price']  -> 4 hasil
    Rp75.000
    Rp150.000
    Rp120.000
    Rp350.000


## 4. Multi-class — perbedaan penting BS4 vs XPath ⚠️

Tiap produk punya class **gabungan**, contoh: `class="product card featured"`.
Kita mau ambil semua elemen yang **punya** class `product`.

- **BS4** `class_="product"` → otomatis cocok kalau `product` ada di antara
  kata-kata class (token match). **Gampang.**
- **XPath** `@class="product"` → cocok hanya kalau class-nya **persis** `"product"`
  → di sini hasilnya **KOSONG** (karena isinya `"product card featured"`).

Solusi XPath yang benar untuk "mengandung token `product`":

```
//*[contains(concat(' ', normalize-space(@class), ' '), ' product ')]
```

Ini menambahkan spasi di kiri-kanan, lalu cek `' product '` agar tidak salah cocok
dengan kata seperti `product-list`.

In [5]:
# BS4: token match -> langsung dapat 4 produk
produk_bs4 = soup.find_all(class_="product")
print("BS4  class_='product'      ->", len(produk_bs4), "produk")

# XPath SALAH: @class harus persis -> 0 hasil
salah = tree.xpath('//div[@class="product"]')
print("XPath @class='product'     ->", len(salah), "hasil (SALAH, kosong)")

# XPath BENAR: cek token dengan concat+contains
benar = tree.xpath(
    "//div[contains(concat(' ', normalize-space(@class), ' '), ' product ')]"
)
print("XPath contains(token)      ->", len(benar), "produk (BENAR)")

# Bonus: hanya produk 'featured'
featured = tree.xpath(
    "//div[contains(concat(' ', normalize-space(@class), ' '), ' featured ')]/h2/text()"
)
show("XPath featured", [t.strip() for t in featured])

BS4  class_='product'      -> 4 produk
XPath @class='product'     -> 0 hasil (SALAH, kosong)
XPath contains(token)      -> 4 produk (BENAR)
XPath featured  -> 1 hasil
    Belajar Python


## 5. Elemen tanpa class → pakai posisi (nth)

Menu navigasi `<li>` tidak punya class. Kita ambil **link menu ke-2** ("Elektronik").

- **BS4** tidak punya "ambil elemen ke-N" di dalam `find`, jadi kita ambil semua
  lalu **index list** (ingat: Python mulai dari 0).
- **XPath** punya posisi bawaan: `[2]` (XPath mulai dari **1**).

| | Sintaks |
| --- | --- |
| **BS4** | `soup.find("ul").find_all("li")[1]` |
| **XPath** | `//ul/li[2]/a` |

In [6]:
# BS4: index ke-1 (elemen ke-2), 0-based
li_kedua = soup.find("ul").find_all("li")[1]
print("BS4  li[index 1]:", li_kedua.find("a").get_text(strip=True))

# XPath: li[2], 1-based
a_kedua = tree.xpath("//ul/li[2]/a/text()")[0]
print("XPath //ul/li[2]:", a_kedua.strip())

BS4  li[index 1]: Elektronik
XPath //ul/li[2]: Elektronik


## 6. Pilih berdasarkan atribut (`data-*`)

Kadang elemen tidak punya class/id yang berguna, tapi punya atribut `data-*`.
Ambil semua produk kategori **buku** lewat `data-category="buku"`.

| | Sintaks |
| --- | --- |
| **BS4** | `soup.find_all(attrs={"data-category": "buku"})` |
| **XPath** | `//*[@data-category="buku"]` |

Mengambil **nilai atribut** (bukan teks): BS4 `el["data-id"]`, XPath `.../@data-id`.

In [7]:
# BS4: filter berdasarkan atribut, lalu ambil data-id + nama
buku_bs4 = soup.find_all(attrs={"data-category": "buku"})
show("BS4  data-category=buku", [(d["data-id"], d.find("h2").text) for d in buku_bs4])

# XPath
buku_xp = tree.xpath('//*[@data-category="buku"]')
show("XPath data-category=buku", [(d.get("data-id"), d.find("h2").text) for d in buku_xp])

# Ambil langsung daftar nilai atribut data-id (semua produk) via XPath
ids = tree.xpath("//div[@data-id]/@data-id")
print("\nSemua data-id (XPath //div[@data-id]/@data-id):", ids)

BS4  data-category=buku  -> 2 hasil
    ('101', 'Belajar Python')
    ('103', 'Data Engineering 101')
XPath data-category=buku  -> 2 hasil
    ('101', 'Belajar Python')
    ('103', 'Data Engineering 101')

Semua data-id (XPath //div[@data-id]/@data-id): ['101', '102', '103', '104']


## 7. aria-label & role (atribut aksesibilitas)

`aria-label` dan `role` sering jadi penyelamat saat elemen tidak punya class/id —
apalagi untuk tombol/ikon. Caranya **sama persis** seperti atribut biasa.

| Target | BS4 | XPath |
| --- | --- | --- |
| Elemen `role="navigation"` | `soup.find(attrs={"role": "navigation"})` | `//*[@role="navigation"]` |
| Tombol dengan aria-label tertentu | `soup.find(attrs={"aria-label": "..."})` | `//*[@aria-label="..."]` |

In [8]:
# role: temukan area navigasi
print("BS4  role=navigation :", soup.find(attrs={"role": "navigation"})["id"])
print("XPath role=navigation:", tree.xpath('//*[@role="navigation"]')[0].get("id"))

# aria-label: tombol 'tambah ke keranjang' untuk Mouse Wireless
label = "Tambah Mouse Wireless ke keranjang"
print("\nBS4  aria-label:", soup.find(attrs={"aria-label": label}).name)
print("XPath aria-label:", tree.xpath(f'//*[@aria-label="{label}"]')[0].tag)

# Ambil semua aria-label tombol Tambah (starts-with) -> lihat juga case 9
labels = tree.xpath('//button/@aria-label')
show("\nSemua aria-label <button>", labels)

BS4  role=navigation : top-nav
XPath role=navigation: top-nav

BS4  aria-label: button
XPath aria-label: button

Semua aria-label <button>  -> 3 hasil
    Tambah Belajar Python ke keranjang
    Tambah Mouse Wireless ke keranjang
    Tambah Data Engineering 101 ke keranjang


## 8. href — ambil link

Tiga kebutuhan umum: **semua link**, **link produk internal** (mulai `/produk/`),
dan **link email** (`mailto:`).

| Target | BS4 | XPath |
| --- | --- | --- |
| Semua href | `soup.find_all("a", href=True)` | `//a/@href` |
| Mulai `/produk/` | `find_all("a", href=re.compile(r"^/produk/"))` | `//a[starts-with(@href,"/produk/")]/@href` |
| Email | `find_all("a", href=re.compile(r"^mailto:"))` | `//a[starts-with(@href,"mailto:")]/@href` |

In [9]:
# Semua href
show("BS4  semua href", [a["href"] for a in soup.find_all("a", href=True)])
show("XPath //a/@href", tree.xpath("//a/@href"))

# Link produk internal (mulai dengan /produk/)
print()
show("BS4  ^/produk/", [a["href"] for a in soup.find_all("a", href=re.compile(r"^/produk/"))])
show("XPath starts-with", tree.xpath('//a[starts-with(@href,"/produk/")]/@href'))

# Link email
print()
show("XPath mailto", tree.xpath('//a[starts-with(@href,"mailto:")]/@href'))

BS4  semua href  -> 9 hasil
    /
    /kategori/buku
    /kategori/elektronik
    https://promo.external.com/diskon
    /produk/101
    /produk/102
    /produk/103
    /produk/104
    mailto:cs@tokokita.id
XPath //a/@href  -> 9 hasil
    /
    /kategori/buku
    /kategori/elektronik
    https://promo.external.com/diskon
    /produk/101
    /produk/102
    /produk/103
    /produk/104
    mailto:cs@tokokita.id

BS4  ^/produk/  -> 4 hasil
    /produk/101
    /produk/102
    /produk/103
    /produk/104
XPath starts-with  -> 4 hasil
    /produk/101
    /produk/102
    /produk/103
    /produk/104

XPath mailto  -> 1 hasil
    mailto:cs@tokokita.id


## 9. Pencocokan sebagian (contains / starts-with / regex)

Kadang kita cuma tahu **sebagian** nilai atribut. Misalnya semua link yang class-nya
mengandung `btn`, atau href yang mengandung `kategori`.

- **BS4**: pakai **regular expression** (`re.compile(...)`) pada nilai atribut.
- **XPath**: pakai fungsi `contains(...)` atau `starts-with(...)`.

> ⚠️ `contains(@class, "btn")` itu pencocokan **substring**, jadi juga cocok dengan
> `"btn-primary"`. Untuk cocok **token** yang tepat, pakai trik `concat` dari case 4.

In [10]:
# Semua link yang class-nya mengandung "btn"
btn_bs4 = soup.find_all("a", class_=re.compile("btn"))
show("BS4  class ~ btn (regex)", [a["class"] for a in btn_bs4])

btn_xp = tree.xpath('//a[contains(@class,"btn")]/@class')
show("XPath contains(@class,'btn')", btn_xp)

# href yang mengandung 'kategori'
print()
show("XPath contains href kategori", tree.xpath('//a[contains(@href,"kategori")]/@href'))

BS4  class ~ btn (regex)  -> 4 hasil
    ['btn', 'btn-primary']
    ['btn', 'btn-primary']
    ['btn', 'btn-primary']
    ['btn', 'btn-disabled']
XPath contains(@class,'btn')  -> 4 hasil
    btn btn-primary
    btn btn-primary
    btn btn-primary
    btn btn-disabled

XPath contains href kategori  -> 2 hasil
    /kategori/buku
    /kategori/elektronik


## 10. Pilih berdasarkan teks

Kadang penanda paling stabil justru **teksnya**. Contoh: cari badge bertuliskan "Habis".

| | Sintaks |
| --- | --- |
| **BS4** (persis) | `soup.find("span", string="Habis")` |
| **BS4** (sebagian) | `soup.find_all("h2", string=re.compile("Engineering"))` |
| **XPath** (persis) | `//span[text()="Habis"]` |
| **XPath** (sebagian) | `//h2[contains(text(),"Engineering")]` |

In [11]:
# Teks persis "Habis"
print("BS4  string='Habis' :", soup.find("span", string="Habis"))
print("XPath text()='Habis':", tree.xpath('//span[text()="Habis"]')[0].text)

# Teks sebagian: nama produk yang mengandung 'Engineering'
print()
hit_bs4 = [h.get_text(strip=True) for h in soup.find_all("h2", string=re.compile("Engineering"))]
show("BS4  regex 'Engineering'", hit_bs4)
hit_xp = [t.strip() for t in tree.xpath('//h2[contains(text(),"Engineering")]/text()')]
show("XPath contains 'Engineering'", hit_xp)

BS4  string='Habis' : <span class="badge">Habis</span>
XPath text()='Habis': Habis

BS4  regex 'Engineering'  -> 1 hasil
    Data Engineering 101
XPath contains 'Engineering'  -> 1 hasil
    Data Engineering 101


## 11. Navigasi: parent / ancestor

Pola umum: temukan elemen yang gampang dikenali (mis. badge "Habis"),
lalu **naik** ke produk pembungkusnya untuk ambil `data-id`/nama.

| | Sintaks |
| --- | --- |
| **BS4** | `el.find_parent("div", class_="product")` |
| **XPath** | `el/ancestor::div[...token ' product '...]` |

> ⚠️ Awas: `contains(@class,"product")` juga cocok dengan `<div class="product-list">`
> (substring!). Pakai trik token `concat` dari case 4 supaya naik ke kartu produk yang benar.

In [12]:
# BS4: dari badge "Habis" -> naik ke div produk
badge = soup.find("span", string="Habis")
produk_habis = badge.find_parent("div", class_="product")
print("BS4  :", produk_habis["data-id"], "-", produk_habis.find("h2").text)

# XPath: dari node badge -> ancestor div product
# Pakai token-match (' product ') supaya tidak ikut ke <div class="product-list">
badge_xp = tree.xpath('//span[text()="Habis"]')[0]
produk_xp = badge_xp.xpath(
    "ancestor::div[contains(concat(' ', normalize-space(@class), ' '), ' product ')]"
)[0]
print("XPath:", produk_xp.get("data-id"), "-", produk_xp.find("h2").text)

BS4  : 104 - Keyboard Mekanik
XPath: 104 - Keyboard Mekanik


## 12. Navigasi: sibling (tetangga)

Pola "label lalu nilai di sebelahnya". Contoh: dari `<h2>` nama produk,
ambil `<span class="price">` yang jadi **saudara berikutnya**.

| | Sintaks |
| --- | --- |
| **BS4** | `h2.find_next_sibling("span", class_="price")` |
| **XPath** | `//h2/following-sibling::span[@class="price"][1]` |

In [13]:
# BS4: dari h2 pertama -> harga di sebelahnya
h2 = soup.find("h2", class_="product-name")
harga_sib = h2.find_next_sibling("span", class_="price")
print("BS4  :", h2.text, "->", harga_sib.text)

# XPath: following-sibling, ambil yang pertama [1]
pasangan = tree.xpath(
    '//h2[@class="product-name"]/following-sibling::span[@class="price"][1]/text()'
)
show("XPath following-sibling (semua produk)", [p.strip() for p in pasangan])

BS4  : Belajar Python -> Rp75.000
XPath following-sibling (semua produk)  -> 4 hasil
    Rp75.000
    Rp150.000
    Rp120.000
    Rp350.000


## 13. Direct child saja (`/` vs `//`)

`<main>` punya `<p>` langsung ("Menampilkan beberapa produk...") **dan** ada `<p>`
di dalam `<footer>` (alamat email). Bagaimana ambil **hanya anak langsung**?

- **BS4**: `recursive=False` → hanya cari di level anak langsung.
- **XPath**: `main/p` (satu garis miring = anak langsung) vs `main//p` (dua garis = semua turunan).

| | Sintaks |
| --- | --- |
| **BS4** | `main.find_all("p", recursive=False)` |
| **XPath** | `//main/p` (langsung) vs `//main//p` (semua) |

In [14]:
main = soup.find("main")

# BS4: anak langsung saja
langsung = main.find_all("p", recursive=False)
show("BS4  recursive=False (anak langsung)", [p.get_text(strip=True) for p in langsung])

# BS4: semua p (termasuk di footer)
semua = main.find_all("p")
show("BS4  semua p (default)", [p.get_text(strip=True) for p in semua])

# XPath: / vs //
print()
show("XPath //main/p (langsung)", [t.strip() for t in tree.xpath("//main/p/text()")])
show("XPath //main//p (semua)", [t.strip() for t in tree.xpath("//main//p//text()")])

BS4  recursive=False (anak langsung)  -> 1 hasil
    Menampilkan beberapa produk pilihan.
BS4  semua p (default)  -> 2 hasil
    Menampilkan beberapa produk pilihan.
    Kontak:cs@tokokita.id

XPath //main/p (langsung)  -> 1 hasil
    Menampilkan beberapa produk pilihan.
XPath //main//p (semua)  -> 3 hasil
    Menampilkan beberapa produk pilihan.
    Kontak:
    cs@tokokita.id


## 14. Kombinasi kondisi + filter numerik

Gabungkan beberapa syarat dengan `and`/`or` di XPath. Di sini XPath punya kelebihan:
bisa **filter angka** langsung dengan `number(...)`.

Contoh: cari produk dengan **harga > 200.000** (data-price), lalu ambil namanya.

- **BS4**: tidak ada operator numerik di `find` → kita **filter di Python**.
- **XPath**: `//span[@class="price"][number(@data-price) > 200000]`.

Plus contoh `and`: link yang **class-nya btn-primary** DAN **href mulai `/produk/10`**.

In [15]:
# Harga > 200.000
# BS4: filter manual di Python
mahal_bs4 = [
    s.find_parent("div").find("h2").text
    for s in soup.find_all("span", class_="price")
    if int(s["data-price"]) > 200000
]
show("BS4  harga>200000 (filter Python)", mahal_bs4)

# XPath: filter numerik inline + naik ke h2 saudara sebelumnya
mahal_xp = tree.xpath(
    '//span[@class="price"][number(@data-price) > 200000]'
    '/preceding-sibling::h2/text()'
)
show("XPath number(@data-price)>200000", [t.strip() for t in mahal_xp])

# Kombinasi AND: class btn-primary DAN href mulai /produk/10
print()
kombinasi = tree.xpath(
    '//a[contains(@class,"btn-primary") and starts-with(@href,"/produk/10")]/@href'
)
show("XPath AND (btn-primary & /produk/10)", kombinasi)

BS4  harga>200000 (filter Python)  -> 1 hasil
    Keyboard Mekanik
XPath number(@data-price)>200000  -> 1 hasil
    Keyboard Mekanik

XPath AND (btn-primary & /produk/10)  -> 3 hasil
    /produk/101
    /produk/102
    /produk/103


## 15. Mini-project: ekstrak semua produk → list of dict

Gabungkan semua teknik: untuk tiap produk ambil **nama, kategori, harga (angka),
link, dan status stok**. Kita tulis versi BS4 dan versi XPath, lalu pastikan hasilnya sama.

Perhatikan penanganan data yang **tidak lengkap** (produk "Keyboard Mekanik" tidak punya
rating, tapi punya badge "Habis").

In [16]:
import pandas as pd

PRODUCT_XPATH = "//div[contains(concat(' ', normalize-space(@class), ' '), ' product ')]"


def parse_bs4():
    hasil = []
    for p in soup.find_all("div", class_="product"):
        badge = p.find("span", class_="badge")
        hasil.append(
            {
                "nama": p.find("h2", class_="product-name").get_text(strip=True),
                "kategori": p["data-category"],
                "harga": int(p.find("span", class_="price")["data-price"]),
                "link": p.find("a", class_="btn")["href"],
                "stok": "habis" if badge and badge.get_text(strip=True) == "Habis" else "tersedia",
            }
        )
    return hasil


def parse_xpath():
    hasil = []
    for p in tree.xpath(PRODUCT_XPATH):
        badge = p.xpath('.//span[@class="badge"]/text()')  # relatif: diawali "."
        hasil.append(
            {
                "nama": p.xpath('.//h2[@class="product-name"]/text()')[0].strip(),
                "kategori": p.get("data-category"),
                "harga": int(p.xpath('.//span[@class="price"]/@data-price')[0]),
                "link": p.xpath('.//a[contains(@class,"btn")]/@href')[0],
                "stok": "habis" if badge and badge[0].strip() == "Habis" else "tersedia",
            }
        )
    return hasil


data_bs4 = parse_bs4()
data_xp = parse_xpath()
assert data_bs4 == data_xp, "Hasil BS4 dan XPath harus sama!"
print("BS4 == XPath ✅  (hasil identik)\n")
pd.DataFrame(data_bs4)

BS4 == XPath ✅  (hasil identik)



,nama,kategori,harga,link,stok
0,Belajar Python,buku,75000,/produk/101,tersedia
1,Mouse Wireless,elektronik,150000,/produk/102,tersedia
2,Data Engineering 101,buku,120000,/produk/103,tersedia
3,Keyboard Mekanik,elektronik,350000,/produk/104,habis


## 16. Cheat Sheet — BeautifulSoup ↔ XPath

| Tujuan | BeautifulSoup | XPath (lxml / Selenium) |
| --- | --- | --- |
| Semua tag `h2` | `soup.find_all("h2")` | `//h2` |
| Berdasarkan id | `soup.find(id="x")` | `//*[@id="x"]` |
| Class (1 token) | `soup.find_all(class_="price")` | `//*[@class="price"]` |
| Class (multi-class) | `soup.find_all(class_="product")` | `//*[contains(concat(' ',normalize-space(@class),' '),' product ')]` |
| Elemen ke-N | `soup.find_all("li")[1]` (0-based) | `(//li)[2]` (1-based) |
| Atribut persis | `soup.find_all(attrs={"data-id":"101"})` | `//*[@data-id="101"]` |
| aria / role | `soup.find(attrs={"role":"navigation"})` | `//*[@role="navigation"]` |
| Ambil nilai atribut | `el["href"]` | `.../@href` |
| Mulai dengan | `re.compile(r"^/produk/")` | `starts-with(@href,"/produk/")` |
| Mengandung | `re.compile("btn")` | `contains(@class,"btn")` |
| Teks persis | `soup.find(string="Habis")` | `//*[text()="Habis"]` |
| Teks sebagian | `string=re.compile("Eng")` | `contains(text(),"Eng")` |
| Parent / ancestor | `el.find_parent("div")` | `ancestor::div` |
| Sibling berikutnya | `el.find_next_sibling("span")` | `following-sibling::span[1]` |
| Anak langsung | `el.find_all("p", recursive=False)` | `el/p` (vs `el//p`) |
| Gabung syarat | (filter di Python) | `[a and b]`, `number(@x) > n` |

## Pitfalls (sering bikin pusing)

- **XPath `@class="..."` harus PERSIS.** Untuk multi-class pakai trik `concat`/`contains`.
- **Index beda awal:** Python list `[0]`, XPath posisi `[1]`.
- **`find()` bisa `None`.** Akses `.text`/`["href"]` ke `None` → error. Cek dulu (lihat badge yang opsional).
- **`contains()` itu substring.** `contains(@class,"btn")` juga match `"btn-primary"`; kalau perlu token persis pakai `concat`.
- **XPath relatif** dari sebuah elemen harus diawali titik: `el.xpath(".//span")`. Tanpa titik (`//span`) akan mencari dari **seluruh dokumen**.
- **`/text()` vs `.text`:** XPath `/text()` mengembalikan string; kalau ada anak tag, teks bisa terpotong (gunakan `.text_content()` di lxml untuk gabungan).

## Latihan

1. Ambil **semua link kategori** (`/kategori/...`) — tulis versi BS4 **dan** XPath.
2. Ambil **nama produk termahal** menggunakan XPath numerik (`number(@data-price)`).
3. Dari teks `"(c) 2026 TokoKita"` di footer, ambil elemen `<span>`-nya lewat pencocokan teks.

Coba dulu sebelum lihat contoh jawaban di bawah.

In [17]:
# --- Contoh jawaban ---

# 1. Semua link kategori
show("BS4  kategori", [a["href"] for a in soup.find_all("a", href=re.compile(r"^/kategori/"))])
show("XPath kategori", tree.xpath('//a[starts-with(@href,"/kategori/")]/@href'))

# 2. Nama produk termahal (XPath murni: "tidak ada harga lain yang lebih besar")
print()
termahal = tree.xpath(
    '//span[@class="price"]'
    '[not(number(@data-price) < number(//span[@class="price"]/@data-price))]'
    '/preceding-sibling::h2/text()'
)
print("Produk termahal (XPath):", [t.strip() for t in termahal])

# 3. Span copyright lewat pencocokan teks
print()
print("BS4  :", soup.find("span", string=re.compile("TokoKita")).get_text(strip=True))
print("XPath:", tree.xpath('//footer//span[contains(text(),"TokoKita")]/text()')[0].strip())

BS4  kategori  -> 2 hasil
    /kategori/buku
    /kategori/elektronik
XPath kategori  -> 2 hasil
    /kategori/buku
    /kategori/elektronik

Produk termahal (XPath): ['Belajar Python', 'Mouse Wireless', 'Data Engineering 101', 'Keyboard Mekanik']

BS4  : (c) 2026 TokoKita
XPath: (c) 2026 TokoKita
